# MedViLL Pre-training — Colab (T4 GPU)

**Yêu cầu:** Runtime → Change runtime type → **T4 GPU**

| Bước | Nội dung |
|------|----------|
| 1 | Kiểm tra GPU |
| 2 | Clone code từ GitHub |
| 3 | Cài dependencies |
| 4 | Upload ảnh X-ray |
| 5 | Kiểm tra data + config |
| 6 | Chạy training |


In [1]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("❌ Không có GPU! Vào Runtime → Change runtime type → T4 GPU")

print(f"✅ GPU   : {torch.cuda.get_device_name(0)}")
print(f"✅ VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"✅ CUDA  : {torch.version.cuda}")
print(f"✅ PyTorch: {torch.__version__}")

RuntimeError: ❌ Không có GPU! Vào Runtime → Change runtime type → T4 GPU

## Bước 2: Clone code từ GitHub

In [ ]:
import os

# ============================================================
# ⚠️  THAY 2 DÒNG NÀY
GITHUB_USERNAME = 'YOUR_USERNAME'   # GitHub username của bạn
REPO_NAME       = 'PKD-NCKH'        # Tên repo trên GitHub
BRANCH          = 'main'             # Tên branch (main hoặc master)
# ============================================================

WORK_DIR = f'/content/{REPO_NAME}'

if os.path.exists(WORK_DIR):
    print("Repo đã tồn tại → pull code mới nhất...")
    os.chdir(WORK_DIR)
    os.system(f'git pull origin {BRANCH}')
else:
    print("Cloning repo từ GitHub...")
    ret = os.system(f'git clone -b {BRANCH} https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git {WORK_DIR}')
    if ret != 0:
        raise RuntimeError(f"❌ Clone thất bại! Kiểm tra GITHUB_USERNAME='{GITHUB_USERNAME}' và REPO_NAME='{REPO_NAME}'")
    os.chdir(WORK_DIR)

os.chdir(WORK_DIR)
print(f"\n✅ Thư mục làm việc: {os.getcwd()}")
os.system('git log --oneline -3')


## Bước 3: Cài dependencies

In [ ]:
%%bash
pip install -q \
    transformers==4.40.0 \
    fuzzywuzzy \
    python-Levenshtein \
    pyyaml \
    tqdm \
    Pillow \
    pandas \
    scikit-learn

echo "✅ Dependencies installed"


## Bước 4: Upload ảnh X-ray vào Colab

Upload file zip chứa ảnh lên Colab rồi giải nén vào đúng thư mục.  
Ảnh cần nằm tại: `data/dataset/open_i/512_3ch/*.jpg`

> Nếu ảnh đã có sẵn trong repo thì **bỏ qua bước này**.


In [ ]:
import os
from google.colab import files

WORK_DIR = f'/content/{REPO_NAME}'
IMG_DIR  = os.path.join(WORK_DIR, 'data/dataset/open_i/512_3ch')

# Nếu ảnh chưa có → upload file zip
if not os.path.exists(IMG_DIR) or len(os.listdir(IMG_DIR)) == 0:
    print("Upload file zip chứa ảnh (512_3ch.zip hoặc tên bất kỳ)...")
    uploaded = files.upload()   # chọn file zip từ máy tính

    for filename in uploaded:
        print(f"Đang giải nén {filename}...")
        os.makedirs(IMG_DIR, exist_ok=True)
        os.system(f'unzip -q "{filename}" -d "{IMG_DIR}"')
        # Nếu unzip tạo thêm 1 cấp thư mục con, di chuyển ảnh lên:
        # os.system(f'mv {IMG_DIR}/512_3ch/* {IMG_DIR}/ && rmdir {IMG_DIR}/512_3ch')
        print(f"✅ Giải nén xong: {len(os.listdir(IMG_DIR))} files trong {IMG_DIR}")
else:
    print(f"✅ Ảnh đã có sẵn: {len(os.listdir(IMG_DIR))} files tại {IMG_DIR}")


## Bước 5: Kiểm tra data + config

In [ ]:
import json, os, yaml, torch

WORK_DIR = f'/content/{REPO_NAME}'
os.chdir(WORK_DIR)

# --- JSONL files ---
print("=== JSONL files ===")
for split in ['Train', 'Valid', 'Test']:
    p = f'data/dataset/openi/{split}.jsonl'
    if os.path.exists(p):
        with open(p) as f:
            n = sum(1 for _ in f)
        print(f"  ✅ {p}: {n} samples")
    else:
        print(f"  ❌ Không thấy: {p}")

# --- Ảnh ---
print("\n=== Ảnh ===")
img_dir = 'data/dataset/open_i/512_3ch'
if os.path.exists(img_dir):
    imgs = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))]
    print(f"  ✅ {img_dir}: {len(imgs)} ảnh")
else:
    print(f"  ❌ Không thấy: {img_dir}  → chạy lại Bước 4 để upload ảnh")

# --- Sample check ---
print("\n=== Sample check ===")
with open('data/dataset/openi/Train.jsonl') as f:
    s = json.loads(f.readline())
img_file = os.path.basename(s.get('img', ''))
full     = os.path.join(img_dir, img_file)
print(f"  img   : {s.get('img')}")
print(f"  text  : {s.get('text','')[:80]}")
print(f"  exists: {'✅' if os.path.exists(full) else '❌'} {full}")

# --- Config ---
print("\n=== Config ===")
cfg_path = 'configs/pretrain.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
if vram_gb >= 14:
    best_bs, best_ga = 64, 2
elif vram_gb >= 10:
    best_bs, best_ga = 32, 4
elif vram_gb >= 6:
    best_bs, best_ga = 16, 8
else:
    best_bs, best_ga = 8, 16

print(f"  VRAM  : {vram_gb:.1f} GB")
print(f"  batch : {best_bs} × accum {best_ga} = effective {best_bs * best_ga}")

changed = False
for key, val in [('batch_size', best_bs), ('gradient_accumulation_steps', best_ga)]:
    if cfg.get(key) != val:
        print(f"  ⚠️  {key}: {cfg.get(key)} → {val}")
        cfg[key] = val
        changed = True

if changed:
    with open(cfg_path, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)
    print("  ✅ Config đã cập nhật")
else:
    print("  ✅ Config OK")


## Bước 6: Chạy Training

Checkpoint lưu tại `/content/output/` (trong Colab).  
> **Lưu ý:** Colab có thể reset — nếu muốn giữ checkpoint lâu dài, mount Drive và đổi `OUTPUT_PATH`.


In [ ]:
import os, subprocess, sys

WORK_DIR    = f'/content/{REPO_NAME}'
OUTPUT_PATH = '/content/output/run1'
os.chdir(WORK_DIR)
os.makedirs(OUTPUT_PATH, exist_ok=True)

print(f"Working dir : {WORK_DIR}")
print(f"Output      : {OUTPUT_PATH}")
print("=" * 60)

result = subprocess.run(
    [
        sys.executable, f"{WORK_DIR}/main.py",
        "--config",      f"{WORK_DIR}/configs/pretrain.yaml",
        "--output_path", OUTPUT_PATH,
    ],
    cwd=WORK_DIR,
)

if result.returncode != 0:
    raise RuntimeError(f"❌ Training thất bại! Exit code: {result.returncode}")
else:
    print("\n✅ Training hoàn tất!")


## (Tuỳ chọn) Theo dõi VRAM — chạy bất cứ lúc nào

In [ ]:
import torch
total    = torch.cuda.get_device_properties(0).total_memory / 1024**3
alloc    = torch.cuda.memory_allocated()     / 1024**3
reserved = torch.cuda.memory_reserved()      / 1024**3
peak     = torch.cuda.max_memory_allocated() / 1024**3
print(f"Total   : {total:.2f} GB")
print(f"Alloc   : {alloc:.2f} GB")
print(f"Reserved: {reserved:.2f} GB")
print(f"Peak    : {peak:.2f} GB")
print(f"Free    : {total - reserved:.2f} GB")